# White Lodge Savings — mini-warehouse analysis

Answers the two questions that arrived by email, and doubles as the starting
point for ad-hoc ones.

**Before running:** `python -m pipeline.run` (builds the warehouse).

| call | what it does |
|---|---|
| `tables()` | everything in the warehouse |
| `columns("marts.fct_claim")` | columns and types of one table |
| `q("select ...")` | SQL to DataFrame |
| `usd(cents)` `pct(fraction)` | formatting |
| `px.bar` `px.line` `px.scatter` `px.imshow` | charts, plain plotly express |

Two idioms used throughout, worth knowing before editing a chart live:

```python
fig.update_xaxes(tickprefix="$", tickformat="~s")            # money axis: $1.2M
fig.update_layout(title=dict(text="...",                     # title + grey note
                             subtitle=dict(text="...")))
```

Money is always **integer cents** in the warehouse. Divide by 100 only when
displaying — `usd()` does that for you.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import plotly.express as px

from analysis.wls import q, tables, columns, usd, pct

px.defaults.template = "plotly_white"
px.defaults.height = 420

tables()

,schema,table,rows
0,marts,dim_date,153
1,marts,dim_drug,49
2,marts,dim_partner,8
3,marts,dim_pharmacy,37
4,marts,dq_rejects,2387
5,marts,fct_claim,41400
6,marts,fct_lookup,176721
7,marts,mart_drug_economics,49
8,marts,mart_funnel_daily,2126
9,marts,mart_partner_performance,8


---
## 0. What survived ingestion

Before any business number: how much of the raw data is usable, and what was
blocked. Without that figure in mind, every total below is a guess.

In [2]:
coverage = q("""
    select
        (select count(*) from raw.claims)                                  as raw_rows,
        (select count(*) from marts.fct_claim)                             as analysable,
        (select count(*) from marts.dq_rejects where source_table='claims') as quarantined
""").iloc[0]

totals = q("""
    select
        count(*)                                   as claims,
        count(*) filter (where is_reverted)        as reverted,
        sum(net_price_cents)                       as gmv,
        sum(net_wls_revenue_cents)                 as wls_revenue,
        count(*) filter (where not has_cost_match) as no_cost
    from marts.fct_claim
""").iloc[0]

pd.DataFrame([
    ("analysable claims",         f"{coverage.analysable:,}"),
    ("coverage",                  pct(coverage.analysable / coverage.raw_rows)),
    ("net GMV",                   usd(totals.gmv)),
    ("White Lodge revenue",       usd(totals.wls_revenue)),
    ("reversal rate",             pct(totals.reverted / totals.claims)),
    ("claims with no NADAC cost", f"{totals.no_cost:,}"),
], columns=["", "the analysable base"])

,,the analysable base
0,analysable claims,"41,400"
1,coverage,96.6%
2,net GMV,$200.7M
3,White Lodge revenue,$183.6k
4,reversal rate,6.6%
5,claims with no NADAC cost,571.0


In [3]:
# Why each row was dropped -- and, more usefully, what kind of problem it is.
# malformed    -> the producer has to fix it; the row never comes back
# ambiguous    -> a human has to decide once (one UUID on two different claims)
# out_of_scope -> nothing to fix; it returns by itself when reference data catches up
q("""
    select defect_class, detected_in, source_table, reject_reason,
           count(*) as rows
    from marts.dq_rejects
    group by 1, 2, 3, 4
    order by 1, 5 desc
""")

,defect_class,detected_in,source_table,reject_reason,rows
0,ambiguous,staging,claims,duplicate_claim_id,281
1,ambiguous,staging,reverts,duplicate_revert_id,26
2,malformed,staging,lookups,unparseable_timestamp,844
3,malformed,staging,claims,non_positive_amount,279
4,malformed,staging,claims,missing_required_field,148
5,malformed,staging,claims,unparseable_timestamp,146
6,malformed,staging,claims,unparseable_number,126
7,malformed,staging,reverts,missing_required_field,13
8,malformed,staging,reverts,unparseable_timestamp,11
9,out_of_scope,intermediate,claims,unknown_npi,460


In [4]:
# The question that the split makes answerable: how much of what we rejected is
# actually recoverable?
q("""
    select defect_class,
           count(*)                                              as rows,
           round(100.0 * count(*) / sum(count(*)) over (), 1)     as pct
    from marts.dq_rejects
    group by 1
    order by 2 desc
""")

,defect_class,rows,pct
0,malformed,1567,65.6
1,out_of_scope,513,21.5
2,ambiguous,307,12.9


In [5]:
# Auditable down to the original text: what exactly was "unreadable"?
q("""
    select record_id, raw_payload
    from marts.dq_rejects
    where reject_reason = 'unparseable_number'
    limit 3
""")

,record_id,raw_payload
0,3f73797f-d5bf-4a91-a1e5-101458d74f28,"{""id"":""3f73797f-d5bf-4a91-a1e5-101458d74f28"",""..."
1,0bc462f0-3b98-4205-8e0c-8f12d3cc4e4e,"{""id"":""0bc462f0-3b98-4205-8e0c-8f12d3cc4e4e"",""..."
2,9e0b5a6b-c24c-47f6-a57c-c40df6ac5189,"{""id"":""9e0b5a6b-c24c-47f6-a57c-c40df6ac5189"",""..."


---
## 1. Dale Cooper — partner mix before renegotiations

> *"For a chain of your choice: who's our most valuable partner, and how do they
> compare to the second-best?"*

The question hides a trap: **"most valuable" by which measure?** By claim volume
and by retained revenue the answer is different, because the commercial terms
range from a flat $1.00 cut to 80% of the fee.

Start with the overall picture, then drop into the chain.

In [6]:
partners = q("""
    select
        partner,
        fee_model,
        lookups, claims,
        net_wls_revenue_cents    as wls_revenue,
        net_partner_payout_cents as payout,
        conversion_rate,
        reversal_rate,
        wls_fee_retention        as retention,
        revenue_cents_per_lookup as revenue_per_lookup
    from marts.mart_partner_performance
    where claims > 0
    order by wls_revenue desc
""")
partners.assign(
    wls_revenue=lambda d: d.wls_revenue.map(usd),
    payout=lambda d: d.payout.map(usd),
    conversion_rate=lambda d: d.conversion_rate.map(pct),
    reversal_rate=lambda d: d.reversal_rate.map(pct),
    retention=lambda d: d.retention.map(pct),
    revenue_per_lookup=lambda d: d.revenue_per_lookup.map(usd),
)

,partner,fee_model,lookups,claims,wls_revenue,payout,conversion_rate,reversal_rate,retention,revenue_per_lookup
0,Kafka Rx,flat,19694,10534,$63.1k,$10.0k,53.5%,5.3%,86.3%,$3.20
1,Hudi Rx,percentage,56625,11104,$37.6k,$37.7k,19.6%,8.0%,50.0%,$0.66
2,Druid Rx,percentage,58993,5789,$31.2k,$7.8k,9.8%,8.1%,80.0%,$0.53
3,Iceberg Rx,flat,13008,3536,$23.8k,$664.60,27.2%,6.0%,97.3%,$1.83
4,Airflow Rx,flat,11190,1715,$11.2k,$0.00,15.3%,8.6%,100.0%,$1.00
5,Flink Rx,percentage,16325,7899,$11.0k,$44.2k,48.4%,5.0%,20.0%,$0.68
6,direct,none,0,823,$5.7k,$0.00,—,7.3%,100.0%,—


In [7]:
# Part-to-whole: of every dollar of pbm_fee a partner originates, how much stays
# with us and how much walks out. This is the comparison the terms obscure.
# Renaming before the melt is what puts readable names in the legend.
mix = (
    partners
    .rename(columns={"wls_revenue": "White Lodge keeps", "payout": "Partner payout"})
    .melt(id_vars="partner",
          value_vars=["White Lodge keeps", "Partner payout"],
          var_name="side", value_name="cents")
    .assign(usd=lambda d: d.cents / 100)
)

fig = px.bar(mix, x="usd", y="partner", color="side", orientation="h",
             labels={"usd": "", "side": "", "partner": ""})
fig.update_yaxes(categoryorder="total ascending")
fig.update_xaxes(tickprefix="$", tickformat="~s")
fig.update_layout(title=dict(
    text="Where the pbm_fee goes, by partner",
    subtitle=dict(text="Net of reversals · Mar–Jul 2026 · "
                       "'direct' is claims that arrived with no lookup, so we keep all of it")))
fig

### Dropping into one chain

The email asks for a specific chain. I pick **`meridian`**, the largest by claim
volume — it's where a renegotiation moves the most money.

In [8]:
chain = "meridian"

by_chain = q(f"""
    select
        partner,
        count(*)                                             as claims,
        sum(net_claim_count)                                 as net_claims,
        sum(net_price_cents)                                 as gmv,
        sum(net_pbm_fee_cents)                               as fee_collected,
        sum(net_partner_fee_cents)                           as payout,
        sum(net_wls_revenue_cents)                           as wls_revenue,
        count(*) filter (where is_reverted) * 1.0 / count(*) as reversal_rate
    from marts.fct_claim
    where chain = '{chain}'
    group by 1
    order by wls_revenue desc
""")
by_chain.assign(
    gmv=lambda d: d.gmv.map(usd),
    fee_collected=lambda d: d.fee_collected.map(usd),
    payout=lambda d: d.payout.map(usd),
    wls_revenue=lambda d: d.wls_revenue.map(usd),
    reversal_rate=lambda d: d.reversal_rate.map(pct),
)

,partner,claims,net_claims,gmv,fee_collected,payout,wls_revenue,reversal_rate
0,Kafka Rx,3253,3052.0,$16.6M,$22.5k,$3.1k,$19.5k,6.2%
1,Hudi Rx,3046,2759.0,$11.6M,$20.2k,$10.1k,$10.1k,9.4%
2,Druid Rx,1524,1364.0,$7.4M,$10.0k,$2.0k,$8.0k,10.5%
3,Iceberg Rx,904,849.0,$4.6M,$6.2k,$169.80,$6.0k,6.1%
4,Airflow Rx,412,372.0,$1.2M,$2.6k,$0.00,$2.6k,9.7%
5,Flink Rx,1764,1659.0,$8.9M,$12.3k,$9.9k,$2.5k,6.0%
6,direct,236,216.0,$956.6k,$1.5k,$0.00,$1.5k,8.5%


In [9]:
fig = px.bar(by_chain.assign(revenue=lambda d: d.wls_revenue / 100),
             x="revenue", y="partner", orientation="h",
             labels={"revenue": "", "partner": ""})
fig.update_yaxes(categoryorder="total ascending")
fig.update_xaxes(tickprefix="$", tickformat="~s")
fig.update_layout(title=dict(
    text=f"White Lodge net revenue in the {chain} chain, by partner",
    subtitle=dict(text="After the partner payout and net of reversals")))
fig

In [10]:
# The full chain x partner picture, so the chain isn't chosen blind.
# imshow on a pivot, not density_heatmap: the data is already aggregated, so
# there is nothing left to bin — and a pivot shows an empty combination as blank
# instead of as a zero.
grid = q("""
    select chain, partner, sum(net_wls_revenue_cents) / 100.0 as revenue
    from marts.fct_claim
    group by 1, 2
""").pivot(index="chain", columns="partner", values="revenue")

fig = px.imshow(grid, color_continuous_scale="Blues", text_auto=",.0f",
                aspect="auto", height=460,
                labels=dict(x="", y="", color="USD"))
fig.update_layout(title=dict(
    text="Revenue retained by White Lodge — chain x partner",
    subtitle=dict(text="USD, net of reversals · darker is more revenue")))
fig

---
## 2. Gordon Cole — where the margin would come from

> *"If we wanted to increase White Lodge's margin, what would you suggest —
> based on what you're seeing in the data?"*

The answer is in one thing that shows up the moment you put price and fee on the
same chart.

In [11]:
sample = q("""
    select price_cents / 100.0 as price, pbm_fee_cents / 100.0 as fee
    from marts.fct_claim
    where not is_reverted
""")

fig = px.scatter(sample, x="price", y="fee", log_x=True, opacity=0.2,
                 labels={"price": "claim price (log scale)", "fee": "pbm_fee charged"})
fig.update_traces(marker=dict(size=3))
fig.update_xaxes(tickprefix="$", tickformat="~s")
fig.update_yaxes(tickprefix="$", tickformat="~s")
fig.update_layout(title=dict(
    text="What we charge has no relationship to the value we intermediate",
    subtitle=dict(text=f"One point per non-reverted claim ({len(sample):,})")))
fig

In [12]:
# The same fact, quantified: take rate collapses as the claim grows.
bands = q("""
    select
        case
            when price_cents < 10000   then '1 · up to $100'
            when price_cents < 1000000 then '2 · $100 to $10k'
            else                            '3 · above $10k'
        end                                         as band,
        count(*)                                    as claims,
        sum(price_cents)                            as gmv,
        sum(pbm_fee_cents)                          as fee,
        sum(pbm_fee_cents) * 1.0 / sum(price_cents) as take_rate
    from marts.fct_claim
    where not is_reverted
    group by 1
    order by 1
""")
bands.assign(gmv=lambda d: d.gmv.map(usd), fee=lambda d: d.fee.map(usd),
             take_rate=lambda d: d.take_rate.map(lambda v: f"{v*100:.4f}%"))

,band,claims,gmv,fee,take_rate
0,1 · up to $100,24143,$561.5k,$180.5k,32.1503%
1,2 · $100 to $10k,12757,$14.5M,$86.9k,0.5982%
2,3 · above $10k,1761,$185.6M,$16.5k,0.0089%


In [13]:
# Two measures on incomparable scales (GMV in millions, take rate in thousandths
# of a percent) => two charts. Never a secondary axis.
# `category descending` puts band 1 at the top, so both charts read top-down in
# the same order as the table above.
fig = px.bar(bands.assign(gmv_usd=lambda d: d.gmv / 100),
             x="gmv_usd", y="band", orientation="h",
             labels={"gmv_usd": "", "band": ""})
fig.update_yaxes(categoryorder="category descending")
fig.update_xaxes(tickprefix="$", tickformat="~s")
fig.update_layout(title=dict(text="Where the financial volume is",
                             subtitle=dict(text="Net GMV by claim value band")))
fig

In [14]:
fig = px.bar(bands.assign(take_bp=lambda d: d.take_rate * 10000),
             x="take_bp", y="band", orientation="h",
             labels={"take_bp": "", "band": ""})
fig.update_yaxes(categoryorder="category descending")
fig.update_layout(title=dict(
    text="Where our compensation is",
    subtitle=dict(text="Take rate in basis points (1 bp = 0.01%) · "
                       "same ordering as the chart above")))
fig

### The biggest number here — and why it is *not* a lever

For every brand drug, NADAC publishes the cost of the equivalent generic, so
we can estimate how much acquisition cost would leave the chain if the same
fill were dispensed as the generic. It comes to **$29.6M**, two orders of
magnitude above everything else in this notebook.

It belongs here, and it does not belong in the margin answer. That $29.6M is
*pharmacy* acquisition cost, not White Lodge revenue — none of it lands on our
P&L. What it is instead is a negotiating asset: demonstrable evidence of value
delivered to plan sponsors. Reading it as a margin lever is the easiest
mistake to make with this dataset, which is why it is called out rather than
ranked.

In [15]:
generics = q("""
    select
        ndc, ndc_description, drug_class,
        net_claims                         as claims,
        net_gmv_cents                      as gmv,
        net_wls_revenue_cents              as wls_revenue,
        generic_substitution_savings_cents as potential_saving,
        reversal_rate
    from marts.mart_drug_economics
    where generic_substitution_savings_cents > 0
    order by potential_saving desc
    limit 10
""")
generics.assign(gmv=lambda d: d.gmv.map(usd),
                wls_revenue=lambda d: d.wls_revenue.map(usd),
                potential_saving=lambda d: d.potential_saving.map(usd),
                reversal_rate=lambda d: d.reversal_rate.map(pct))

,ndc,ndc_description,drug_class,claims,gmv,wls_revenue,potential_saving,reversal_rate
0,61958220101,EPCLUSA 400 MG-100 MG TABLET,brand,602.0,$39.6M,$3.3k,$24.3M,7.7%
1,10631011831,ABSORICA 40 MG CAPSULE,brand,1100.0,$3.2M,$7.0k,$2.4M,7.2%
2,00078050161,EXELON 4.6 MG/24HR PATCH,brand,1622.0,$2.6M,$11.6k,$2.2M,7.6%
3,50419045304,CLIMARA 0.075 MG/DAY PATCH,brand,1072.0,$1.5M,$874.23,$413.1k,5.9%
4,00002418430,EVISTA 60 MG TABLET,brand,550.0,$262.6k,$1.6k,$225.9k,7.1%
5,70165001530,ADZENYS XR-ODT 9.4 MG TABLET,brand,572.0,$850.8k,$4.2k,$115.6k,5.5%


In [16]:
fig = px.bar(generics.head(6).assign(saving=lambda d: d.potential_saving / 100),
             x="saving", y="ndc_description", orientation="h",
             labels={"saving": "", "ndc_description": ""})
fig.update_yaxes(categoryorder="total ascending")
fig.update_xaxes(tickprefix="$", tickformat="~s")
fig.update_layout(title=dict(
    text="Acquisition cost that generic substitution would remove",
    subtitle=dict(text="Gap between brand NADAC and the equivalent generic, "
                       "at dispensed volume")))
fig

### The second real lever: reversals

A reversal is revenue that was earned and handed back. Unlike margin that never
existed, this one is recoverable — $13,127, 7.1% of everything White Lodge
retains, at a median of 9.5 days from fill to reversal.

In [17]:
reversals = q("""
    select
        sum(wls_fee_cents) filter (where is_reverted) as revenue_lost,
        sum(wls_fee_cents)                            as revenue_potential,
        median(hours_to_revert)                           as hours_to_revert
    from marts.fct_claim
""").iloc[0]

pd.DataFrame([
    ("revenue lost to reversals", usd(reversals.revenue_lost)),
    ("% of potential revenue",    pct(reversals.revenue_lost / reversals.revenue_potential)),
    ("median time to reversal",   f"{reversals.hours_to_revert / 24:.1f} days"),
], columns=["", "what reversals cost"])

,,what reversals cost
0,revenue lost to reversals,$13.1k
1,% of potential revenue,6.7%
2,median time to reversal,9.5 days


In [18]:
# Cohort rate: of the claims *filled* that week, how many were ever reverted.
# mart_funnel_daily also carries reverts_on_day, keyed on the reversal date —
# the two only agree over the full period.
#
# 'direct' and 'unknown' are excluded: they are the synthetic members of
# dim_partner (no lookup / no partner on the lookup), not partners you can call
# about a reversal rate.
weekly = q("""
    select week_start, partner,
           sum(claims) as claims,
           sum(claims_filled_then_reverted) as reverted
    from marts.mart_funnel_daily
    where partner not in ('unknown', 'direct')
    group by 1, 2
    order by 1
""")
weekly["rate"] = weekly.reverted / weekly.claims

fig = px.line(weekly, x="week_start", y="rate", color="partner",
              labels={"rate": "", "week_start": "", "partner": ""})
fig.update_yaxes(tickformat=".0%")
fig.update_layout(title=dict(
    text="Reversal rate by partner, by fill week",
    subtitle=dict(text="Cohort rate · flat across all of them: "
                       "a structural cost, not a one-off incident")))
fig

---
## Scratch area

Room for whatever arrives live. The shortest path is almost always
`marts.fct_claim` on its own — it already carries `chain`, `partner`, `channel`
and `drug_class` denormalised for exactly this.

In [19]:
# The three quality measures beside the commercial terms, so the last section is
# provable rather than asserted. Synthetic members are excluded: `direct` has no
# lookups to convert, and `unknown` never converted at all.
quality = q("""
    select partner, fee_model,
           conversion_rate, reversal_rate, wls_fee_retention,
           net_wls_revenue_cents as wls_revenue
    from marts.mart_partner_performance
    where not is_synthetic
    order by wls_revenue desc
""")
quality.assign(conversion_rate=lambda d: d.conversion_rate.map(pct),
               reversal_rate=lambda d: d.reversal_rate.map(pct),
               wls_fee_retention=lambda d: d.wls_fee_retention.map(pct),
               wls_revenue=lambda d: d.wls_revenue.map(usd))


,partner,fee_model,conversion_rate,reversal_rate,wls_fee_retention,wls_revenue
0,Kafka Rx,flat,53.5%,5.3%,86.3%,$63.1k
1,Hudi Rx,percentage,19.6%,8.0%,50.0%,$37.6k
2,Druid Rx,percentage,9.8%,8.1%,80.0%,$31.2k
3,Iceberg Rx,flat,27.2%,6.0%,97.3%,$23.8k
4,Airflow Rx,flat,15.3%,8.6%,100.0%,$11.2k
5,Flink Rx,percentage,48.4%,5.0%,20.0%,$11.0k


---
## Reading this out loud

Four numbers above are easy to state and easy to get wrong under questioning.
Each one is a figure with a condition attached, and the condition is the part
worth saying.

### 1. The $29.6M is not ours

NADAC publishes the cost of the equivalent generic alongside every brand drug,
so the gap between the two, at the volume we actually dispensed, is $29.6M.

That money is the **pharmacy's acquisition cost** — what the pharmacy paid for
the drug, not what we earned on it. If every one of those fills switched to the
generic tomorrow, White Lodge revenue would not move by a cent: our `pbm_fee`
does not depend on what the drug cost. So it is evidence of value delivered,
something to put in front of a plan sponsor, and not a margin lever. It is the
largest figure in this notebook, which is exactly what makes it the most
dangerous one to quote without that sentence attached.

Two details behind it: the $29.6M comes from **six drugs**, and Epclusa alone is
$24.3M of it. Three of our nine brand drugs — Tremfya, Trulicity, Advair — have
no published generic in NADAC at all and contribute zero, including Tremfya,
which is the single biggest driver of GMV in the margin section above.

### 2. Reversals: two denominators, both correct

$13,127 of fee was collected and handed back. Whether that is 6.7% or 7.1%
depends entirely on what you divide by:

* **6.7%** = $13,127 / $196,772 — everything we *would have* kept if nothing had
  reversed. The share of the opportunity that was lost.
* **7.1%** = $13,127 / $183,645 — what we actually kept. How much bigger the book
  would be if we recovered it.

Neither is wrong; they answer different questions. Say which one you mean.

### 3. A reversal has two dates, and inside a month they disagree

A claim filled in March can be reverted in July, and **26% of reversals land in a
different month than the fill they cancel**. So "how many reversals in March" has
two true answers:

* **Cohort — 524.** Of the claims *filled* in March, how many were ever reverted.
  Keyed on the fill date. This is the measure for comparing partners, because it
  asks whether a partner sends fills that stick.
* **Activity — 358.** How many reversals actually *happened* in March. Keyed on
  the reversal date. This is the measure for "what did we hand back last month?"

The two agree only over the whole period (2,739 each). March is 524 against 358;
July is 595 against 800, high because July collects the tail of every earlier
month. `mart_funnel_daily` carries both and calls neither one `reverted_claims`.
The chart above is the cohort measure.

The consequence, which is better said than asked about: because the money is
measured on the fill date, **history restates**. A reversal arriving today
removes revenue from the month of the original fill, not from today. March is
"March as we understand it now", not "March as we reported it in April".

### 4. Flink Rx is our best partner and our worst deal

The margin answer recommends renegotiating Flink, which is easy to hear as
"Flink is underperforming". The table above says otherwise. Flink converts at
**48.4%**, second of six, and has the **lowest cohort reversal rate of any
partner — 5.0%**, against 8.6% for Airflow and 8.0% for Hudi. Operationally it is
the strongest relationship we have. The only thing wrong with it is the 80/20
split, which is why it ranks last on what White Lodge keeps.

That cuts both ways, and both halves belong in the room:

* It is the cleanest demonstration in the dataset that **terms decide value** —
  not volume, and not funnel quality. Rank partners by claims or by conversion
  and Flink is top two; rank them by what we keep and it is last.
* It is also the reason to move carefully. The **+$16,563** estimate assumes the
  volume survives the renegotiation, and this is the partner we would least like
  to lose.
